# Project 1



In [ ]:
from pyspark.sql.types import *
import pyspark.sql.functions as f

from pyspark import SparkContext
from pyspark.sql import SparkSession
 
spark = SparkSession.builder.appName("my_project_1").getOrCreate()
sc = spark.sparkContext

In [ ]:
# Read a CSV into a dataframe
# There is a smarter version, that will first check if there is a Parquet file and use it
def load_csv_file(filename, schema):
  # Reads the relevant file from distributed file system using the given schema

  allowed_files = {'Daily program data': ('Daily program data', "|"),
                   'demographic': ('demographic', "|")}

  if filename not in allowed_files.keys():
    print(f'You were trying to access unknown file \"{filename}\". Only valid options are {allowed_files.keys()}')
    return None

  filepath = allowed_files[filename][0]
  dataPath = f"dbfs:/mnt/coursedata2024/fwm-stb-data/{filepath}"
  delimiter = allowed_files[filename][1]

  df = spark.read.format("csv")\
    .option("header","false")\
    .option("delimiter",delimiter)\
    .schema(schema)\
    .load(dataPath)
  return df

# This dict holds the correct schemata for easily loading the CSVs
schemas_dict = {'Daily program data':
                  StructType([
                    StructField('prog_code', StringType()),
                    StructField('title', StringType()),
                    StructField('genre', StringType()),
                    StructField('air_date', StringType()),
                    StructField('air_time', StringType()),
                    StructField('Duration', FloatType())
                  ]),
                'viewing':
                  StructType([
                    StructField('device_id', StringType()),
                    StructField('event_date', StringType()),
                    StructField('event_time', IntegerType()),
                    StructField('mso_code', StringType()),
                    StructField('prog_code', StringType()),
                    StructField('station_num', StringType())
                  ]),
                'viewing_full':
                  StructType([
                    StructField('mso_code', StringType()),
                    StructField('device_id', StringType()),
                    StructField('event_date', IntegerType()),
                    StructField('event_time', IntegerType()),
                    StructField('station_num', StringType()),
                    StructField('prog_code', StringType())
                  ]),
                'demographic':
                  StructType([StructField('household_id',StringType()),
                    StructField('household_size',IntegerType()),
                    StructField('num_adults',IntegerType()),
                    StructField('num_generations',IntegerType()),
                    StructField('adult_range',StringType()),
                    StructField('marital_status',StringType()),
                    StructField('race_code',StringType()),
                    StructField('presence_children',StringType()),
                    StructField('num_children',IntegerType()),
                    StructField('age_children',StringType()), #format like range - 'bitwise'
                    StructField('age_range_children',StringType()),
                    StructField('dwelling_type',StringType()),
                    StructField('home_owner_status',StringType()),
                    StructField('length_residence',IntegerType()),
                    StructField('home_market_value',StringType()),
                    StructField('num_vehicles',IntegerType()),
                    StructField('vehicle_make',StringType()),
                    StructField('vehicle_model',StringType()),
                    StructField('vehicle_year',IntegerType()),
                    StructField('net_worth',IntegerType()),
                    StructField('income',StringType()),
                    StructField('gender_individual',StringType()),
                    StructField('age_individual',IntegerType()),
                    StructField('education_highest',StringType()),
                    StructField('occupation_highest',StringType()),
                    StructField('education_1',StringType()),
                    StructField('occupation_1',StringType()),
                    StructField('age_2',IntegerType()),
                    StructField('education_2',StringType()),
                    StructField('occupation_2',StringType()),
                    StructField('age_3',IntegerType()),
                    StructField('education_3',StringType()),
                    StructField('occupation_3',StringType()),
                    StructField('age_4',IntegerType()),
                    StructField('education_4',StringType()),
                    StructField('occupation_4',StringType()),
                    StructField('age_5',IntegerType()),
                    StructField('education_5',StringType()),
                    StructField('occupation_5',StringType()),
                    StructField('polit_party_regist',StringType()),
                    StructField('polit_party_input',StringType()),
                    StructField('household_clusters',StringType()),
                    StructField('insurance_groups',StringType()),
                    StructField('financial_groups',StringType()),
                    StructField('green_living',StringType())
                  ])
}

# Read demogrphic data


In [ ]:
%%time
# demographic data filename is 'demographic' 
demo_df = load_csv_file('demographic', schemas_dict['demographic'])

# Read Daily program data

In [ ]:
%%time 
from pyspark.sql.functions import col, to_date, date_format, substring, row_number, countDistinct
from pyspark.sql.window import Window

# daily_program data filename is 'Daily program data' 
daily_prog_df = load_csv_file('Daily program data', schemas_dict['Daily program data'])

#we don't want a duplicates rows 
daily_prog_df = daily_prog_df.distinct() 

#we don't want a situations in which there is same program(identified by prog_code) with different title/genre so bellow are the steps : 

# Identify prog_code with multiple distinct titles or genres 
conflicting_prog_codes = daily_prog_df.groupBy('prog_code') \
    .agg(
        countDistinct('title').alias('distinct_title_count'),
        countDistinct('genre').alias('distinct_genre_count') 
    ) \
    .filter((col('distinct_title_count') > 1) | (col('distinct_genre_count') > 1)) \
    .select('prog_code') 

# Filter out conflicting prog_code rows
daily_prog_df_filtered = daily_prog_df.join(conflicting_prog_codes, on='prog_code', how='left_anti') 

# Assign single title and genre for each conflicting prog_code 
windowSpec = Window.partitionBy('prog_code').orderBy('title') #the orderBy doesn't relevant for our needs
single_title_genre_per_progcode = daily_prog_df.join(conflicting_prog_codes, on = 'prog_code') \
                                        .withColumn("row_num", row_number().over(windowSpec)) \
                                        .filter(col("row_num") == 1) \
                                        .select('prog_code', 'title', 'genre') 

# Then, we leave for each confilcting prog_code only the compatible rows(with its decided (title, genre))
compatible_confilcting_progcodes = daily_prog_df.join(conflicting_prog_codes, on='prog_code') \
                                            .join(single_title_genre_per_progcode, on=['prog_code','title','genre']) 
                                                                                                           
# Combine non-conflicting and resolved conflicting households 
daily_prog_df_resolved = daily_prog_df_filtered.union(compatible_confilcting_progcodes) 
#over 

# we use that version of daily_prog_df_resolved only when needed 
daily_prog_df_with_date = daily_prog_df_resolved.withColumn('air_date', to_date(col('air_date'), 'yyyyMMdd'))
daily_prog_df_with_date= daily_prog_df_with_date.filter(daily_prog_df_with_date.air_date.isNotNull())
daily_prog_df_with_date= daily_prog_df_with_date.filter(daily_prog_df_with_date.air_time.isNotNull())
daily_prog_df_with_air_day = daily_prog_df_with_date.withColumn('air_day', date_format(col('air_date'), 'EEEE'))
new_daily_prog_df = daily_prog_df_with_air_day.withColumn('air_hour', substring(col('air_time'), 1, 2))


# Read viewing data

In [ ]:
# Sample of 10 Million viewing entries

dataPath = f"dbfs:/viewing_10M"
viewing10m_df = spark.read.format("csv")\
    .option("header","true")\
    .option("delimiter",",")\
    .schema(schemas_dict['viewing_full'])\
    .load(dataPath) 
# we don't want unnecessary rows 
viewing10m_df = viewing10m_df.distinct() 


# Read reference data

In [ ]:
%%time
# reference data is stored in parquet for your convinence.
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, countDistinct

ref_df = spark.read.parquet('dbfs:/refxml_new_parquet')
print(f'ref_df contains {ref_df.count()} records!')

updated_ref_df = ref_df.withColumn('dma_contains_z', col('dma').contains('z'))  
 
# we don't want a situation in which there is same device to 2 different households or same device with 
# same household (Unnecessary rows) 

new_ref_df = updated_ref_df.dropDuplicates(subset=['device_id']) 

# we don't want a situation in which there is same household with 2 different DMA so bellow are the steps:

# Identify households associated with multiple DMAs
conflicting_households = new_ref_df.groupBy('household_id') \
                        .agg(countDistinct('dma').alias('DMA_count')) \
                        .filter(col('DMA_count') > 1) \
                        .select('household_id') 

# Filter out rows with conflicting household_id and different dma 
ref_df_filtered = new_ref_df.join(conflicting_households, on='household_id', how='left_anti') 

# Assign a single DMA per conflicting household 
windowSpec2 = Window.partitionBy('household_id').orderBy('device_id') #the orderBy doesn't relevant for our needs
single_dma_per_household = new_ref_df.join(conflicting_households, on='household_id') \
                                    .withColumn("row_num", row_number().over(windowSpec2)) \
                                    .filter(col("row_num") == 1) \
                                    .select("household_id", "dma") 
            
# Then, we leave for each confilcting household only the compatible rows(with its decided dma )
compatible_conflicting_households = new_ref_df.join(conflicting_households, on='household_id') \
                                            .join(single_dma_per_household, on=['household_id', 'dma']) 
                                                                                                    
# Combine non-conflicting and resolved conflicting households
ref_df_resolved = ref_df_filtered.union(compatible_conflicting_households) 
#over 



In [ ]:
# Q 2.1 
import re
from pyspark.sql.functions import array_contains, split, when 
from pyspark.sql.window import Window 
from pyspark.sql.functions import split, explode, count, avg, col, when 

# Extract and Filter Unique DMA and Devices
dma_device = ref_df_resolved.select('device_id', 'dma')
dma_device = dma_device.filter(col('device_id').isNotNull() & 
                               col('dma').isNotNull() & 
                               (col('dma') != 'Unknown'))

# Get the Top 10 DMAs by the Number of Devices
top_10_dma = dma_device.groupBy(col('dma')) \
             .agg(count('*').alias('device_num')) \
             .orderBy('device_num', ascending=False) \
             .limit(10) \
             .select('dma') 

def clean_dma_name(dma_name):
    return re.sub(r'[^a-zA-Z0-9_]', '', dma_name.replace(' ', '_')) 

# Function to Process Each DMA and Display Genres Viewing Count
def table_dma(dma_row, dma_num):
    dma_name = dma_row['dma'] 
    cleaned_dma_name = clean_dma_name(dma_name)

    # Extract Unique Devices for the Given DMA
    devices = ref_df_resolved.select('device_id').where(col('dma') == dma_name).distinct()
    filt_devices = devices.filter(col('device_id').isNotNull()) 

    # Join Viewing Data with Devices to Get Unique Programs
    programs = viewing10m_df.join(filt_devices, on='device_id').select('prog_code').filter(col('prog_code').isNotNull()).distinct()
    
    # Join Daily Program Data with Unique Programs to Get relevant Genres 
    join_for_find_genres = daily_prog_df_resolved.join(programs, on='prog_code').filter(col('genre').isNotNull())

    # Split the Genres that appear on one program into Separate Rows, and Apply Distinct
    explode_split = join_for_find_genres.select('prog_code', explode(split('genre', ',')).alias('genre')).distinct()
    
    # Group by Genre and Count Viewing Entries 
    genres_viewing = explode_split.groupBy(col('genre')).agg(count('*').alias('viewing entries')).orderBy('viewing entries', ascending=False)
    
    if dma_num in [1,5,9]: 
        displayHTML(f"<h2>Top 10 Genres (ordered by popularity) for DMA {dma_num}: {dma_name}</h2>")
        display(genres_viewing.limit(10)) 
    
    genres_viewing.write.mode('overwrite').csv(f"project1_part21_{cleaned_dma_name}_000000000_000000000.csv", header=True) 

# Apply the Function to Selected DMAs (1st, 5th, 9th)
i = 1 
for dma in top_10_dma.collect():
    table_dma(dma, i) 
    i += 1

In [ ]:
#Q 2.2

from pyspark.sql.functions import max, when, col, avg, count, split, explode, array_contains, size, concat_ws
import re
from pyspark.sql.window import Window
from pyspark.sql import SparkSession



def clean_dma_name(dma_name):
    return re.sub(r'[^a-zA-Z0-9_]', '', dma_name.replace(' ', '_'))

# Select relevant columns from demo_df and convert 'income' values to integers
relevant_demo = demo_df.select('household_id', 'net_worth', 'income')
relevant_demo = relevant_demo.withColumn(
    'income',
    when(col('income') == 'A', 10)
    .when(col('income') == 'B', 11)
    .when(col('income') == 'C', 12)
    .when(col('income') == 'D', 13)
    .otherwise(col('income').cast('int'))
)

# Compute the maximum values for 'net_worth' and 'income' across all data
max_net_worth_alldata = relevant_demo.select(max('net_worth')).collect()[0][0]
max_income_alldata = relevant_demo.select(max('income')).collect()[0][0]

# Extract and filter unique DMA and household_id from ref_df
dma_household = ref_df.select('household_id', 'dma').distinct()
dma_household = dma_household.filter(col('household_id').isNotNull() & col('dma').isNotNull())

# Join relevant_demo with dma_household to get net_worth and compute average net_worth per DMA
dma_networth = relevant_demo.join(dma_household, on='household_id')
dma_networth = dma_networth.select('dma', 'net_worth')
dma_networth = dma_networth.filter(col('net_worth').isNotNull())
dma_networth = dma_networth.filter(col('dma').isNotNull() & (col('dma') != 'Unknown'))
dma_networth = dma_networth.groupBy('dma').agg(avg('net_worth').alias('avg_net_worth'))


# Join relevant_demo with dma_household to get income and compute average income per DMA
dma_income = relevant_demo.join(dma_household, on='household_id')
dma_income = dma_income.select('income', 'dma')
dma_income = dma_income.filter(col('income').isNotNull())
dma_income = dma_income.groupBy('dma').agg(avg('income').alias('avg_income'))

# Combine average net_worth and average income into dma_wealthscore DataFrame and compute wealth score
dma_wealthscore = ref_df.select('dma').distinct()
dma_wealthscore = dma_wealthscore.join(dma_income, on='dma')
dma_wealthscore = dma_wealthscore.join(dma_networth, on='dma')
dma_wealthscore = dma_wealthscore.filter(col('avg_income').isNotNull() & col('avg_net_worth').isNotNull())
dma_wealthscore = dma_wealthscore.withColumn(
    'wealth_score',
    col('avg_net_worth') / max_net_worth_alldata + col('avg_income') / max_income_alldata
).orderBy('wealth_score', ascending=False) \
             .limit(10) \
             .select('dma', 'wealth_score')


# Define a function to extract the top 11 genres for a given DMA
def genres_list(dma_row):
    dma_name = dma_row['dma']
    # Extract unique devices for the given DMA
    devices = ref_df.select('device_id').where(col('dma') == dma_name).distinct()
    devices = devices.filter(col('device_id').isNotNull())

    # Join viewing data with devices to get unique programs
    programs = viewing10m_df.join(devices, on='device_id')
    programs = programs.select('prog_code').distinct()
    programs = programs.filter(col('prog_code').isNotNull())

    # Join daily program data with unique programs to get genres
    genres_viewing = daily_prog_df.join(programs, on='prog_code')

    # Split genres into separate rows, apply distinct and filter out null values
    genres_viewing = genres_viewing.select('prog_code', explode(split('genre', ',')).alias('genre')).distinct()
    genres_viewing = genres_viewing.filter(col('prog_code').isNotNull() & col('genre').isNotNull())
    
    # Group by genre and count viewing entries, then order by count
    genres_viewing = genres_viewing.groupBy(col('genre')).agg(count('*').alias('viewing entries')).orderBy('viewing entries', ascending=False)

    # Select top 11 genres that haven't been used yet
    top_11_genres = [row['genre'] for row in genres_viewing.filter(~col('genre').isin(used_genres)).select('genre').limit(11).collect()]

    return dma_name, top_11_genres

# Initialize lists to store genre data
dma_genres_list = []
top_11_genres = []
used_genres = []

# Iterate over each DMA to get top 11 genres
for dma in dma_wealthscore.collect():
    dma_name, top_11_genres = genres_list(dma)
    # Ensure the top_11_genres list has exactly 11 elements
    if len(top_11_genres) < 11:
        # Pad with None to ensure the list has 11 elements
        top_11_genres.extend([None] * (11 - len(top_11_genres)))
    dma_genres_list.append((dma_name, top_11_genres))
    used_genres.extend(filter(None, top_11_genres))


# Convert top_11_genres from list to a comma-separated string
dma_genres_list_str = [(dma_name, ','.join(filter(None, genres))) for dma_name, genres in dma_genres_list]

dma_genres = spark.createDataFrame(dma_genres_list, ["dma", "ordered_list_of_genres"])

# Replace empty lists with null
dma_genres = dma_genres.withColumn(
    "ordered_list_of_genres",
    when(size(col("ordered_list_of_genres")) == 0, None).otherwise(col("ordered_list_of_genres"))
)

# Join dma_wealthscore with dma_genres to include genre information
dma_wealthscore_genre = dma_wealthscore.join(dma_genres, on='dma', how='left')
dma_wealthscore_genre = dma_wealthscore_genre.orderBy('wealth_score', ascending=False)

# Show the resulting DataFrame
display(dma_wealthscore_genre)

dma_num=1
for row in dma_wealthscore_genre.collect(): 
    dma_name = row['dma'] 
    cleaned_dma_name = clean_dma_name(dma_name) 
    dma_df = dma_wealthscore_genre.filter(col('dma') == dma_name).select('wealth_score', 'ordered_list_of_genres') 
    exploded_dma_df = dma_df.withColumn('ordered_genres', explode(col('ordered_list_of_genres')))

    if dma_num in [1, 5, 9]:
        displayHTML(f"<h2>Top 10 Genres (ordered by popularity) for DMA {dma_num}: {dma_name}</h2>")
        display(exploded_dma_df.select('ordered_genres'))

    concatenated_dma_df = dma_df.withColumn('ordered_list_of_genres', concat_ws(',', col('ordered_list_of_genres')))
    concatenated_dma_df.write.mode('overwrite').csv(f"project1_part22_{cleaned_dma_name}_000000000_000000000.csv", header=True)
    dma_num+=1



